In [ ]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('..')) # Go up one level to the project root and add it to the path
from src.data.ingest import *

# If you just want to see the whole data config dictionary
#from src.utils.config import load_config
# data_config = load_config("data.yaml")

In [ ]:
#TODO write def to import in src/data/ingest.py
#TODO use config/data.yaml to add the ondrive path and stuff

brfss_df = pd.read_sas("~/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/Hasui, Shota W.'s files - BRFSS/BRFSS/LLCP2024.XPT", format='xport')

In [ ]:
cols = ['_STATE', 'ADDEPEV3', '_LLCPWT'] 
df = brfss_df[cols].copy()

# --- Print prevalence of excluded values ---
total = len(df)
missing = df['ADDEPEV3'].isna().sum()
dont_know = (df['ADDEPEV3'] == 7).sum()
refused = (df['ADDEPEV3'] == 9).sum()

print("--- Excluded Data (ADDEPEV3) ---")
print(f"Missing:    {missing} ({(missing/total)*100:.2f}%)")
print(f"Don't Know: {dont_know} ({(dont_know/total)*100:.2f}%)")
print(f"Refused:    {refused} ({(refused/total)*100:.2f}%)")
print("--------------------------------\n")

# --- Process and calculate prevalence ---
df = df[df['ADDEPEV3'].isin([1, 2])].copy()

df['dep_flag'] = np.where(df['ADDEPEV3'] == 1, 1, 0)
df['weighted_cases'] = df['dep_flag'] * df['_LLCPWT']

# Added include_groups=False to silence the FutureWarning
prevalence = df.groupby('_STATE').apply(
    lambda x: x['weighted_cases'].sum() / x['_LLCPWT'].sum(),
    include_groups=False
).reset_index(name='depression_prevalence')

print(prevalence.head())